# COVID-19 Chest X-Ray Detection — Full Pipeline
**Coventry University | 7156CEM | Channabasavanna Santosh Pawate**

### Steps:
1. Install dependencies
2. Upload & prepare dataset
3. Train ViT-B/16 model
4. Evaluate (accuracy, F1, confusion matrix, ROC)
5. XAI — Attention Rollout heatmaps
6. Test with your own X-ray images
7. Download trained model

## ✅ Step 1: Check GPU & Install Dependencies

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('CUDA version:', torch.version.cuda)

In [ ]:
!pip install -q transformers==4.40.0 timm scikit-learn matplotlib seaborn opencv-python-headless Pillow tqdm

## ✅ Step 2: Upload & Prepare Dataset

> Upload your dataset zip file. Structure should be:
> `data/train/COVID-19/`, `data/train/Normal/`, `data/train/Viral Pneumonia/` etc.

**Option A:** Upload from your PC
**Option B:** Download directly from Kaggle (faster)

In [ ]:
# OPTION B: Download from Kaggle (recommended - faster)
# Upload your kaggle.json first
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()  # upload kaggle.json

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d amanullahasraf/covid19-pneumonia-normal-chest-xraypa-dataset -p /content --unzip
!ls /content/COVID19_Pneumonia_Normal_Chest_Xray_PA_Dataset/

In [ ]:
import os, shutil, random
from pathlib import Path

random.seed(42)
SRC = '/content/COVID19_Pneumonia_Normal_Chest_Xray_PA_Dataset'
DST = '/content/data'

class_map = {'covid': 'COVID-19', 'normal': 'Normal', 'pneumonia': 'Viral Pneumonia'}

for split in ['train', 'val', 'test']:
    for cls in class_map.values():
        os.makedirs(f'{DST}/{split}/{cls}', exist_ok=True)

for src_cls, dst_cls in class_map.items():
    imgs = sorted(os.listdir(f'{SRC}/{src_cls}'))
    random.shuffle(imgs)
    n = len(imgs)
    n_train, n_val = int(n*0.70), int(n*0.15)
    splits = {'train': imgs[:n_train], 'val': imgs[n_train:n_train+n_val], 'test': imgs[n_train+n_val:]}
    for split, files_ in splits.items():
        for f in files_:
            shutil.copy(f'{SRC}/{src_cls}/{f}', f'{DST}/{split}/{dst_cls}/{f}')
        print(f'{split}/{dst_cls}: {len(files_)} images')

print('\nDataset ready!')

## ✅ Step 3: Train the Model (ViT-B/16)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import ViTForImageClassification, ViTImageProcessor
import numpy as np, time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ['COVID-19', 'Normal', 'Viral Pneumonia']
DATA_DIR = '/content/data'
MODEL_ID = 'google/vit-base-patch16-224-in21k'
BATCH_SIZE = 32
print('Using device:', DEVICE)

In [ ]:
# Data transforms
processor = ViTImageProcessor.from_pretrained(MODEL_ID)
mean, std = processor.image_mean, processor.image_std

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_ds = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_tf)
val_ds   = datasets.ImageFolder(f'{DATA_DIR}/val',   transform=val_tf)
test_ds  = datasets.ImageFolder(f'{DATA_DIR}/test',  transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('Train:', len(train_ds), '| Val:', len(val_ds), '| Test:', len(test_ds))
print('Class index map:', train_ds.class_to_idx)

In [ ]:
# Load pretrained ViT
model = ViTForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=3,
    id2label={i: c for i, c in enumerate(CLASSES)},
    label2id={c: i for i, c in enumerate(CLASSES)},
    ignore_mismatched_sizes=True,
    attn_implementation='eager'
).to(DEVICE)
print('Model loaded.')

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(pixel_values=imgs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss/total, correct/total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(pixel_values=imgs).logits
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss/total, correct/total

criterion = nn.CrossEntropyLoss()
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

In [ ]:
# Stage 1: Train classifier head only (frozen backbone)
print('=== Stage 1: Head only (5 epochs) ===')
for param in model.vit.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

for epoch in range(1, 6):
    t = time.time()
    tl, ta = train_epoch(model, train_loader, optimizer, criterion)
    vl, va = eval_epoch(model, val_loader, criterion)
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    print(f'  Epoch {epoch:02d}/5 | Loss {tl:.4f}/{vl:.4f} | Acc {ta:.4f}/{va:.4f} | {time.time()-t:.0f}s')

In [ ]:
# Stage 2: Fine-tune full model
print('=== Stage 2: Full fine-tune (10 epochs) ===')
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

best_val_acc = 0
for epoch in range(1, 11):
    t = time.time()
    tl, ta = train_epoch(model, train_loader, optimizer, criterion)
    vl, va = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    print(f'  Epoch {epoch:02d}/10 | Loss {tl:.4f}/{vl:.4f} | Acc {ta:.4f}/{va:.4f} | {time.time()-t:.0f}s')
    if va > best_val_acc:
        best_val_acc = va
        model.save_pretrained('/content/vit_covid_final')
        print(f'    *** Best model saved (val_acc={va:.4f}) ***')

print(f'\nTraining complete! Best val accuracy: {best_val_acc:.4f}')

## ✅ Step 4: Evaluate — Accuracy, F1, Confusion Matrix, ROC

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F

# Load best model
model = ViTForImageClassification.from_pretrained(
    '/content/vit_covid_final',
    attn_implementation='eager'
).to(DEVICE)
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        logits = model(pixel_values=imgs).logits
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

all_probs = np.array(all_probs)
print(classification_report(all_labels, all_preds, target_names=CLASSES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix — Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve, auc
y_bin = label_binarize(all_labels, classes=[0,1,2])
colors = ['blue','green','red']
plt.figure(figsize=(8,6))
for i, (cls, col) in enumerate(zip(CLASSES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:,i], all_probs[:,i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=col, label=f'{cls} (AUC={roc_auc:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — One vs Rest')
plt.legend()
plt.tight_layout()
plt.savefig('/content/roc_curve.png', dpi=150)
plt.show()

In [ ]:
# Training History Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss'])+1)
ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
ax1.plot(epochs, history['val_loss'],   'r-', label='Val Loss')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax1.axvline(x=5.5, color='gray', linestyle='--', label='Stage 2 start')
ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
ax2.plot(epochs, history['val_acc'],   'r-', label='Val Acc')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
ax2.axvline(x=5.5, color='gray', linestyle='--', label='Stage 2 start')
plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150)
plt.show()

## ✅ Step 5: XAI — Attention Rollout Heatmaps

In [ ]:
import cv2
from PIL import Image

def attention_rollout(model, img_tensor):
    model.eval()
    attentions = []
    hooks = []
    for layer in model.vit.encoder.layer:
        hooks.append(layer.attention.attention.register_forward_hook(
            lambda m, i, o: attentions.append(o[1].detach().cpu())
        ))
    with torch.no_grad():
        output = model(pixel_values=img_tensor.unsqueeze(0).to(DEVICE))
    for h in hooks: h.remove()

    result = torch.eye(attentions[0].shape[-1])
    for attn in attentions:
        attn_avg = attn.squeeze(0).mean(0)
        attn_avg = attn_avg + torch.eye(attn_avg.shape[0])
        attn_avg = attn_avg / attn_avg.sum(dim=-1, keepdim=True)
        result = torch.matmul(attn_avg, result)

    mask = result[0, 1:].reshape(14, 14).numpy()
    mask = (mask - mask.min()) / (mask.max() - mask.min())
    mask = cv2.resize(mask, (224, 224))
    return mask, output.logits.softmax(1).squeeze().cpu().numpy()

def show_heatmap(img_path):
    img = Image.open(img_path).convert('RGB').resize((224,224))
    img_np = np.array(img)
    img_tensor = val_tf(img)

    mask, probs = attention_rollout(model, img_tensor)
    pred_idx = probs.argmax()
    pred_cls = CLASSES[pred_idx]
    confidence = probs[pred_idx]

    heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.6 * img_np + 0.4 * heatmap).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np); axes[0].set_title('Original X-Ray'); axes[0].axis('off')
    axes[1].imshow(heatmap); axes[1].set_title('Attention Heatmap'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title(f'Overlay\nPred: {pred_cls} ({confidence:.1%})'); axes[2].axis('off')

    colors = ['red' if c==pred_cls else 'gray' for c in CLASSES]
    plt.tight_layout()
    plt.show()
    print(f'Prediction: {pred_cls} ({confidence:.1%})')
    for cls, prob in zip(CLASSES, probs):
        print(f'  {cls}: {prob:.1%}')

print('Heatmap function ready!')

In [ ]:
# Test on a random image from test set
import random as rnd
for cls in CLASSES:
    test_class_dir = f'{DATA_DIR}/test/{cls}'
    sample = rnd.choice(os.listdir(test_class_dir))
    print(f'\n--- Testing {cls} sample ---')
    show_heatmap(f'{test_class_dir}/{sample}')

## ✅ Step 6: Test With Your Own X-Ray Image

In [ ]:
# Upload your own X-ray image and test it
from google.colab import files
print('Upload an X-ray image:')
uploaded = files.upload()
for fname in uploaded:
    show_heatmap(fname)

## ✅ Step 7: Download Trained Model

In [ ]:
# Zip and download the trained model
import shutil
shutil.make_archive('/content/vit_covid_final', 'zip', '/content/vit_covid_final')
files.download('/content/vit_covid_final.zip')
print('Model downloaded! Place it in model/saved/vit_covid_final/ in your project.')